# Ultra-Low-Disk FPS Evaluation Workflow: 1-by-1 Download, Render & Instant Cleanup

This notebook automates **strict 1-by-1 rendering FPS evaluation** with **zero upfront bulk downloads**:
1. **3DGS** (using `gaussian-splatting_backup/render.py`)
2. **FastGS** (using `FastGS_backup_v2/render.py`)
3. **Specular-Gaussians** (using `Specular-Gaussians_backup_v2/render.py`)
4. **Spec-FastGS (Ours)** (using `spec-fastgs/render.py`)

### Strict 1-by-1 Low-Disk Pipeline:
For **EACH individual method & dataset**:
- **Step 1**: Download **ONLY THAT 1 archive** from Hugging Face with HF Token & Proxy bypass.
- **Step 2**: Extract archive into temporary local folder.
- **Step 3**: Execute CUDA-synchronized rendering benchmarks (`render.py`) for test scenes.
- **Step 4**: Parse FPS, record metrics, and **INSTANTLY DELETE** extracted files & zip cache before moving to the next dataset!
- **Fault Tolerance**: If any download or CUDA render error occurs, a `[WARNING]` is logged, resources are cleaned up, and execution **seamlessly continues** to the next dataset!


In [ ]:
import os
import sys
import ssl
import json
import glob
import time
import shutil
import zipfile
import warnings
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# 1. Ensure UTF-8 console output
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# 2. Load Hugging Face Token (from environment variable or hf_token.json)
BASE_DIR = Path(__file__).parent.parent if '__file__' in locals() else Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
token_file = BASE_DIR / "hf_token.json"

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and token_file.exists():
    try:
        with open(token_file, "r") as f:
            HF_TOKEN = json.load(f).get("HF_KEY")
    except Exception:
        pass

if not HF_TOKEN:
    print("[NOTICE] HF_TOKEN not found in environment or hf_token.json.")
    print("Please ensure 'hf_token.json' exists in workspace root with {'HF_KEY': 'your_token'}.")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN

# 3. Bypass SSL verification & handle Corporate Proxy for BOSCH / server networks
ssl._create_default_https_context = ssl._create_unverified_context
import requests
from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter('ignore', InsecureRequestWarning)
old_merge = requests.Session.merge_environment_settings
requests.Session.merge_environment_settings = lambda self, url, proxies, stream, verify, cert: \
    {**old_merge(self, url, proxies, stream, verify, cert), 'verify': False}

# 4. Verify huggingface_hub
try:
    from huggingface_hub import HfApi, hf_hub_download, login
except ImportError:
    print("Installing huggingface_hub...")
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
    from huggingface_hub import HfApi, hf_hub_download, login

if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("[SUCCESS] Authenticated with Hugging Face Hub successfully!")
    except Exception as e:
        print(f"[WARNING] HF Login warning: {e}")

STORAGE_DIR = BASE_DIR / "storage"
HF_CACHE_DIR = STORAGE_DIR / "hf_downloads"
RESULTS_ROOT = BASE_DIR / "result_mipnerf_specular"
KETQUA_DIR = BASE_DIR / "ketqua"

for d in [STORAGE_DIR, HF_CACHE_DIR, RESULTS_ROOT, KETQUA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Workspace Root : {BASE_DIR.resolve()}")
print(f"Active Device  : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


In [ ]:
# Dataset & Method Hugging Face Mapping
HF_DATASETS = {
    "Specular-Gaussians": {
        "mipnerf360": {
            "repo_id": "DiBiay/specular_gaussians-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "temp_spec_gaussian_mipnerf360"
        },
        "specular": {
            "repo_id": "DiBiay/specular_gaussians-synthetic_specular-result",
            "filename": "synthetic_specular.zip",
            "target_dir": RESULTS_ROOT / "temp_spec_gaussian_specular"
        }
    },
    "Spec-FastGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/spec-fastgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "temp_spec_fastgs_mipnerf360"
        },
        "specular": {
            "repo_id": "DiBiay/spec-fastgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "temp_spec_fastgs_specular"
        }
    },
    "FastGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/fastgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "temp_fastgs_mipnerf360"
        },
        "specular": {
            "repo_id": "DiBiay/fastgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "temp_fastgs_specular"
        }
    },
    "3DGS": {
        "mipnerf360": {
            "repo_id": "DiBiay/3dgs-mipnerf360-result",
            "filename": "images4.zip",
            "target_dir": RESULTS_ROOT / "temp_3dgs_mipnerf360"
        },
        "specular": {
            "repo_id": "DiBiay/3dgs-synthetic-specular-result",
            "filename": "result.zip",
            "target_dir": RESULTS_ROOT / "temp_3dgs_specular"
        }
    }
}

CODEBASE_PATHS = {
    "3DGS": BASE_DIR / "gaussian-splatting_backup",
    "FastGS": BASE_DIR / "FastGS_backup_v2",
    "Specular-Gaussians": BASE_DIR / "Specular-Gaussians_backup_v2",
    "Spec-FastGS": BASE_DIR / "spec-fastgs"
}

SCENES_MIPNERF360 = ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]
SCENES_SPECULAR = ["ashtray", "dishes", "headphone", "jupyter", "lock", "plane", "record", "teapot"]

print("Ultra-low-disk HF mapping loaded successfully.")


In [ ]:
def download_and_extract_single(method_name, dataset_type):
    cfg = HF_DATASETS[method_name][dataset_type]
    repo_id = cfg["repo_id"]
    filename = cfg["filename"]
    target_dir = Path(cfg["target_dir"])
    target_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n---> [STEP 1/3] Downloading {method_name} ({dataset_type}): {repo_id}/{filename} ...")
    try:
        kwargs = {"repo_id": repo_id, "filename": filename, "repo_type": "dataset", "cache_dir": str(HF_CACHE_DIR)}
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN
            
        zip_path = hf_hub_download(**kwargs)
        print(f"[STEP 1/3] Downloaded archive to cache: {zip_path}")
        
        print(f"---> [STEP 2/3] Extracting archive into: {target_dir}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"[STEP 2/3] Extracted successfully to {target_dir.name}")
        return zip_path, target_dir
    except Exception as e:
        print(f"[WARNING] Could not download or extract {repo_id}/{filename}: {e}")
        print(f"Skipping {method_name} ({dataset_type}) ...")
        return None, target_dir

def find_model_scene_path(base_target_dir, scene_name):
    p = Path(base_target_dir)
    if not p.exists():
        return None
    candidates = [
        p / scene_name,
        p / f"{p.name}-result" / scene_name,
        p / "spec-fastgs-mipnerf360-result-images4" / scene_name,
        p / "spec-fastgs-synthetic-specular-result" / scene_name,
        p / "fastgs-mipnerf360-result" / scene_name,
        p / "fastgs-synthetic-specular-result" / scene_name,
        p / "3dgs-mipnerf360-result" / scene_name,
        p / "3dgs-synthetic-specular-result" / scene_name,
    ]
    for c in candidates:
        if c.exists() and ((c / "point_cloud").exists() or (c / "results.json").exists()):
            return c
    matches = list(p.rglob(scene_name))
    for m in matches:
        if m.is_dir() and ((m / "point_cloud").exists() or (m / "results.json").exists()):
            return m
    return p / scene_name

def cleanup_single(zip_path, target_dir):
    print(f"---> [STEP 3/3] Instant Cleanup for current method...")
    try:
        if target_dir and Path(target_dir).exists():
            print(f"   Deleting extracted directory: {target_dir.name} ...")
            shutil.rmtree(target_dir, ignore_errors=True)
        if zip_path and Path(zip_path).exists():
            print(f"   Deleting downloaded zip cache: {Path(zip_path).name} ...")
            try:
                os.remove(zip_path)
            except Exception:
                pass
        # Also clear any residual cache files in HF_CACHE_DIR
        if HF_CACHE_DIR.exists():
            shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
            HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f"   [WARNING] Cleanup warning: {e}")
    print("[STEP 3/3] Disk space freed completely before next method.\n")


In [ ]:
def evaluate_single_method_fps(method_name, dataset_type, scenes):
    fps_results = {}
    zip_path, target_dir = None, None
    try:
        # 1. Download & Extract 1 archive ONLY when this function is called
        zip_path, target_dir = download_and_extract_single(method_name, dataset_type)
        if not target_dir or not target_dir.exists():
            print(f"[WARNING] Target directory missing for {method_name} ({dataset_type}). Moving to next dataset...")
            return fps_results
            
        code_dir = CODEBASE_PATHS[method_name]
        render_py = code_dir / "render.py"
        if not render_py.exists():
            print(f"[WARNING] render.py not found in {code_dir}. Skipping...")
            return fps_results
        
        print(f"---> Running CUDA Rendering FPS Benchmark: {method_name} | {dataset_type} ...")
        for scene in scenes:
            try:
                model_path = find_model_scene_path(target_dir, scene)
                if not model_path or not model_path.exists():
                    print(f" [WARNING] Scene directory/point cloud not found for {scene:12s}. Skipping...")
                    continue
                    
                cmd = [sys.executable, str(render_py), "-m", str(model_path), "--skip_train"]
                print(f" Benchmarking {scene:12s} ...", end=" ", flush=True)
                res = subprocess.run(cmd, cwd=str(code_dir), capture_output=True, text=True)
                
                # Parse FPS from results.json
                res_json_path = model_path / "results.json"
                fps_val = None
                if res_json_path.exists():
                    try:
                        with open(res_json_path, "r") as f:
                            d = json.load(f).get("ours_30000", {})
                            fps_val = d.get("FPS")
                    except Exception:
                        pass
                    
                if fps_val is not None:
                    print(f"FPS: {fps_val:.2f}")
                    fps_results[scene] = round(fps_val, 2)
                else:
                    print(f"[WARNING] Could not parse FPS for {scene}")
            except Exception as scene_err:
                print(f"\n [WARNING] Exception while rendering {scene}: {scene_err}. Continuing to next scene...")
                
    except Exception as method_err:
        print(f"\n[WARNING] Global exception during {method_name} ({dataset_type}) benchmark: {method_err}")
        print(f"Continuing execution pipeline seamlessly...")
    finally:
        # 2. Always IMMEDIATELY clean up downloaded & extracted data before returning
        cleanup_single(zip_path, target_dir)
        
    return fps_results

print("Single-dataset downloader + renderer + instant cleaner registered successfully.")


In [ ]:
all_fps_data = {"mipnerf360": {}, "specular": {}}
methods_list = ["3DGS", "FastGS", "Specular-Gaussians", "Spec-FastGS"]

# 1. Mip-NeRF 360 Benchmark Loop (1 by 1: Download -> Render FPS -> Instant Delete)
print("\n======================================================")
print("  STARTING MIP-NERF 360 LOW-DISK FPS BENCHMARK LOOP")
print("======================================================")
for method in methods_list:
    try:
        all_fps_data["mipnerf360"][method] = evaluate_single_method_fps(method, "mipnerf360", SCENES_MIPNERF360)
    except Exception as e:
        print(f"[WARNING] Unhandled error benchmarking {method} (mipnerf360): {e}. Moving forward...")

# 2. Synthetic Specular Benchmark Loop (1 by 1: Download -> Render FPS -> Instant Delete)
print("\n======================================================")
print(" STARTING SYNTHETIC SPECULAR LOW-DISK FPS BENCHMARK LOOP")
print("======================================================")
for method in methods_list:
    try:
        all_fps_data["specular"][method] = evaluate_single_method_fps(method, "specular", SCENES_SPECULAR)
    except Exception as e:
        print(f"[WARNING] Unhandled error benchmarking {method} (specular): {e}. Moving forward...")


In [ ]:
def build_fps_table(ds_name, scenes):
    rows = []
    for scene in scenes:
        row = {"Scene": scene}
        for method in methods_list:
            row[method] = all_fps_data[ds_name].get(method, {}).get(scene, None)
        rows.append(row)
    df = pd.DataFrame(rows)
    
    # Add Average Row
    avg_row = {"Scene": "Average"}
    for method in methods_list:
        vals = [v for v in df[method] if v is not None]
        avg_row[method] = round(np.mean(vals), 2) if vals else None
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    return df

df_fps_mip = build_fps_table("mipnerf360", SCENES_MIPNERF360)
df_fps_spec = build_fps_table("specular", SCENES_SPECULAR)

print("\n======================================================")
print("         MIP-NERF 360 RENDERING FPS SUMMARY")
print("======================================================")
display(df_fps_mip)

print("\n======================================================")
print("       SYNTHETIC SPECULAR RENDERING FPS SUMMARY")
print("======================================================")
display(df_fps_spec)

# Export to CSV & Excel in ketqua/
df_fps_mip.to_csv(KETQUA_DIR / "fps_mipnerf360.csv", index=False, encoding="utf-8-sig")
df_fps_spec.to_csv(KETQUA_DIR / "fps_specular.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(KETQUA_DIR / "fps_summary.xlsx") as writer:
    df_fps_mip.to_excel(writer, sheet_name="mipnerf360", index=False)
    df_fps_spec.to_excel(writer, sheet_name="specular", index=False)

print(f"\n[SUCCESS] Low-disk FPS benchmark tables saved to {KETQUA_DIR.resolve()}")
